# 📊 LIDC-IDRI Dataset Profiling & Feasibility Survey (Google Colab)

Notebook này phục vụ **khảo sát toàn diện đặc trưng, phân bổ thống kê, số lượng bệnh nhân và dung lượng lưu trữ** của tập dữ liệu **LIDC-IDRI** (The Lung Image Database Consortium and Image Database Resource Initiative) trước khi tải dữ liệu về máy.

---

### 🎯 Các câu hỏi nghiên cứu cốt lõi cần giải quyết:
1. **Có bao nhiêu bệnh nhân trong dataset? Bao nhiêu bệnh nhân có ca chụp (Study) là CT scan?**
2. **Nếu CHỈ TẢI các chuỗi ảnh cắt lớp CT (bỏ qua X-quang phẳng DX/CR và các báo cáo SEG/SR), thì toàn bộ dataset nặng chính xác bao nhiêu GB?**
3. **Phân bổ số lượng lát cắt (slices/images) và dung lượng của từng ca CT như thế nào?**
4. **Đa dạng về thiết bị chụp (Hãng máy, Model máy) ra sao (ảnh hưởng đến Scanner Bias trong huấn luyện mô hình MambaVision)?**
5. **Chiến lược và dự toán thời gian tải dữ liệu theo từng giai đoạn thử nghiệm (10 bệnh nhân $\to$ 100 $\to$ 500 $\to$ Toàn bộ dataset).**

---
> 💡 **Cơ chế hoạt động**: Notebook kết nối trực tiếp **TCIA Public NBIA API v1** để kéo toàn bộ siêu dữ liệu (metadata) của 15,116 chuỗi ảnh về phân tích (chỉ tốn khoảng ~12 MB tải metadata, **không tải bất kỳ ảnh DICOM nặng nào ở bước này**).


---
## 1. Cài đặt thư viện & Cấu hình môi trường
Sử dụng các thư viện chuẩn phân tích dữ liệu và trực quan hóa y tế: `pandas`, `numpy`, `matplotlib`, `seaborn`, `requests`.


In [ ]:
# 1. Cài đặt thư viện
!pip -q install requests pandas matplotlib seaborn numpy

from pathlib import Path
import json
import time
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Cấu hình giao diện biểu đồ chuyên nghiệp
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['figure.dpi'] = 120
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

print("✅ Thư viện sẵn sàng!")


---
## 2. Truy vấn siêu dữ liệu toàn bộ Dataset từ TCIA NBIA API v1
Chúng ta sử dụng endpoint `getSeries?Collection=LIDC-IDRI&format=json` để lấy thông tin của toàn bộ các chuỗi ảnh trong collection.
Để tối ưu, kết quả sẽ được lưu vào file cache `lidc_series_metadata.json` trên Colab để không phải gọi lại API nhiều lần.


In [ ]:
# 2. Truy vấn siêu dữ liệu (Zero-Image Metadata Query)
BASE_URL = "https://services.cancerimagingarchive.net/nbia-api/services/v1"
CACHE_FILE = Path("lidc_series_metadata.json")

if CACHE_FILE.exists():
    print(f"🔄 Đang đọc metadata từ file cache: {CACHE_FILE}...")
    with open(CACHE_FILE, "r", encoding="utf-8") as f:
        series_data = json.load(f)
    print(f"✅ Đã tải xong {len(series_data)} records từ cache!")
else:
    print("🌐 Đang truy vấn metadata từ TCIA Public NBIA API v1 (mất khoảng 45-60s)...")
    url = f"{BASE_URL}/getSeries"
    params = {"Collection": "LIDC-IDRI", "format": "json"}

    t0 = time.time()
    resp = requests.get(url, params=params, timeout=180, headers={"User-Agent": "Mozilla/5.0"})
    resp.raise_for_status()
    series_data = resp.json()
    elapsed = time.time() - t0

    with open(CACHE_FILE, "w", encoding="utf-8") as f:
        json.dump(series_data, f)

    size_mb = CACHE_FILE.stat().st_size / (1024**2)
    print(f"✅ Hoàn thành truy vấn trong {elapsed:.2f}s! Dung lượng metadata: {size_mb:.2f} MB")
    print(f"Tổng số chuỗi ảnh (series records): {len(series_data):,}")


---
## 3. Tiền xử lý & Chuẩn hóa kiểu dữ liệu
Chuyển đổi dữ liệu thô sang DataFrame Pandas, chuẩn hóa các trường số lượng ảnh (`ImageCount`) và dung lượng tệp (`FileSize` sang MB/GB).


In [ ]:
# 3. Chuẩn hóa dữ liệu sang Pandas DataFrame
df = pd.DataFrame(series_data)

# Ép kiểu dữ liệu
df["ImageCount"] = pd.to_numeric(df["ImageCount"], errors="coerce").fillna(0).astype(int)
df["FileSize"] = pd.to_numeric(df["FileSize"], errors="coerce").fillna(0).astype(int)
df["FileSize_MB"] = df["FileSize"] / (1024**2)
df["FileSize_GB"] = df["FileSize"] / (1024**3)

print("Số dòng dữ liệu (Series):", len(df))
print("Các cột thông tin có sẵn:")
print(list(df.columns))
display(df.head(3))


---
## 4. Phân tích phân bổ phương thức chụp (Modalities)
Khảo sát các phương thức chụp có trong LIDC-IDRI:
- **CT**: Computed Tomography (Ảnh cắt lớp lồng ngực 3D)
- **DX**: Digital Radiography (Ảnh X-quang phẳng 2D)
- **CR**: Computed Radiography (Ảnh X-quang số hóa)
- **SEG**: DICOM Segmentation (Mặt nạ phân vùng nốt phổi của 4 bác sĩ)
- **SR**: Structured Report (Bản đánh giá đặc trưng lâm sàng của 4 bác sĩ)


In [ ]:
# 4. Bảng tổng hợp các Modality trong toàn bộ LIDC-IDRI
modality_summary = df.groupby("Modality").agg(
    SeriesCount=("SeriesInstanceUID", "count"),
    PatientCount=("PatientID", "nunique"),
    TotalImages=("ImageCount", "sum"),
    TotalSize_GB=("FileSize_GB", "sum")
).reset_index()

total_size_all = modality_summary["TotalSize_GB"].sum()
total_images_all = modality_summary["TotalImages"].sum()
total_series_all = modality_summary["SeriesCount"].sum()

modality_summary["Series_Pct"] = (modality_summary["SeriesCount"] / total_series_all) * 100
modality_summary["Size_Pct"] = (modality_summary["TotalSize_GB"] / total_size_all) * 100

modality_summary = modality_summary.sort_values(by="TotalSize_GB", ascending=False)

print("="*85)
print(f"{'BẢNG THỐNG KÊ CHI TIẾT CÁC MODALITY TRONG LIDC-IDRI':^85}")
print("="*85)
for _, r in modality_summary.iterrows():
    print(f"Modality: {r['Modality']:<5} | Series: {r['SeriesCount']:>6,} ({r['Series_Pct']:>5.1f}%) | Bệnh nhân: {r['PatientCount']:>5,} | Ảnh: {r['TotalImages']:>8,} | Dung lượng: {r['TotalSize_GB']:>6.2f} GB ({r['Size_Pct']:>5.1f}%)")
print("="*85)
print(f"{'TỔNG CỘNG':<15} | Series: {total_series_all:>6,} (100.0%) | Bệnh nhân: {df['PatientID'].nunique():>5,} | Ảnh: {total_images_all:>8,} | Dung lượng: {total_size_all:>6.2f} GB (100.0%)")
print("="*85)


In [ ]:
# Trực quan hóa so sánh Modality: Số lượng Series vs. Dung lượng thực tế
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Biểu đồ 1: Số lượng series theo Modality
colors = ['#2563eb', '#059669', '#d97706', '#dc2626', '#7c3aed']
sns.barplot(data=modality_summary, x="Modality", y="SeriesCount", palette="Blues_r", ax=axes[0])
axes[0].set_title("Số lượng Series theo Modality (Tỷ lệ áp đảo của SEG/SR)", fontsize=12, fontweight="bold")
axes[0].set_ylabel("Số lượng Series (log scale)")
axes[0].set_yscale("log")
for p in axes[0].patches:
    val = int(p.get_height())
    axes[0].annotate(f'{val:,}', (p.get_x() + p.get_width() / 2., p.get_height()),
                    ha='center', va='bottom', fontsize=9, xytext=(0, 3), textcoords='offset points')

# Biểu đồ 2: Tỷ lệ dung lượng lưu trữ (GB) theo Modality
axes[1].pie(
    modality_summary["TotalSize_GB"],
    labels=modality_summary["Modality"],
    autopct='%1.1f%%',
    startangle=140,
    colors=['#3b82f6', '#f59e0b', '#10b981', '#6366f1', '#ec4899'],
    explode=[0.05 if m == "CT" else 0.1 for m in modality_summary["Modality"]]
)
axes[1].set_title("Tỷ lệ phân bổ Dung lượng lưu trữ (CT chiếm ~94%)", fontsize=12, fontweight="bold")

plt.tight_layout()
plt.show()


---
## 5. Trả lời câu hỏi 1: Có bao nhiêu bệnh nhân có Study là CT scan?

Khảo sát chi tiết số lượng bệnh nhân có CT, số ca chụp CT trên mỗi bệnh nhân, và mức độ chồng chéo với các ca chụp X-quang.


In [ ]:
# 5. Khảo sát bệnh nhân có CT scan
total_patients = df["PatientID"].nunique()
ct_df = df[df["Modality"] == "CT"].copy()
ct_patients = ct_df["PatientID"].nunique()

print(f"📊 KẾT QUẢ KHẢO SÁT:")
print(f"• Tổng số bệnh nhân trong toàn bộ LIDC-IDRI : {total_patients:,} bệnh nhân")
print(f"• Số bệnh nhân có Study chụp CT              : {ct_patients:,} bệnh nhân ({ct_patients/total_patients*100:.1f}%)")
print(f"• Số bệnh nhân KHÔNG có CT                  : {total_patients - ct_patients} bệnh nhân")

# Kiểm tra phân bố số lượng series CT trên mỗi bệnh nhân
ct_per_patient = ct_df["PatientID"].value_counts()
pts_with_1_ct = (ct_per_patient == 1).sum()
pts_with_2_ct = (ct_per_patient == 2).sum()
pts_with_more_ct = (ct_per_patient > 2).sum()

print(f"\n📈 Phân bố số ca CT trên từng bệnh nhân:")
print(f"  - Số bệnh nhân có ĐÚNG 1 ca CT : {pts_with_1_ct:,} ({pts_with_1_ct/ct_patients*100:.2f}%)")
print(f"  - Số bệnh nhân có 2 ca CT      : {pts_with_2_ct} ({pts_with_2_ct/ct_patients*100:.2f}%)")
print(f"  - Số bệnh nhân có >2 ca CT     : {pts_with_more_ct}")

multi_ct_ids = ct_per_patient[ct_per_patient > 1].index.tolist()
print(f"  - Danh sách ID 8 bệnh nhân có 2 ca CT: {multi_ct_ids}")
print(f"\nTổng số chuỗi ảnh CT (CT Series) cần xử lý: {len(ct_df):,} series")


In [ ]:
# Biểu đồ: Mức độ trùng lặp giữa bệnh nhân CT và bệnh nhân có X-quang (DX/CR)
dx_patients = set(df[df["Modality"].isin(["DX", "CR"])]["PatientID"])
ct_patient_set = set(ct_df["PatientID"])

both_ct_and_xray = len(ct_patient_set.intersection(dx_patients))
ct_only_patients = len(ct_patient_set - dx_patients)

plt.figure(figsize=(7, 4))
bars = plt.bar(
    ["Chỉ có CT scan\n(Không có X-quang)", "Có cả CT scan\nvà ảnh X-quang"],
    [ct_only_patients, both_ct_and_xray],
    color=['#2563eb', '#f59e0b'],
    width=0.5
)
plt.ylabel("Số lượng bệnh nhân")
plt.title("Phân loại bệnh nhân: CT thuần túy vs. CT kèm X-quang đối chiếu", fontsize=11, fontweight="bold")
for b in bars:
    plt.annotate(f'{b.get_height():,} bệnh nhân\n({b.get_height()/total_patients*100:.1f}%)',
                 (b.get_x() + b.get_width() / 2., b.get_height() / 2),
                 ha='center', va='center', color='white', fontweight='bold', fontsize=10)
plt.ylim(0, 850)
plt.tight_layout()
plt.show()


---
## 6. Trả lời câu hỏi 2: Khi CHỈ LẤY Study CT thôi thì Dataset nặng bao nhiêu?

So sánh chi tiết dung lượng lưu trữ của toàn bộ tập dữ liệu gốc so với khi chỉ lọc lấy các chuỗi ảnh CT.


In [ ]:
# 6. Khảo sát dung lượng CT
total_dataset_gb = df["FileSize_GB"].sum()
ct_only_gb = ct_df["FileSize_GB"].sum()
non_ct_gb = total_dataset_gb - ct_only_gb

pct_ct = (ct_only_gb / total_dataset_gb) * 100
saved_gb = non_ct_gb

print("="*75)
print(f"{'SO SÁNH DUNG LƯỢNG LƯU TRỮ (STORAGE REQUIREMENT)':^75}")
print("="*75)
print(f"• Dung lượng TOÀN BỘ DATASET (15,116 Series) : {total_dataset_gb:>8.2f} GB (100.0%)")
print(f"• Dung lượng KHI CHỈ LẤY CT (1,018 Series)   : {ct_only_gb:>8.2f} GB ({pct_ct:.1f}%)")
print(f"• Dung lượng tiết kiệm được (Bỏ DX/CR/SEG/SR) : {saved_gb:>8.2f} GB ({100-pct_ct:.1f}%)")
print(f"• Số lượng file DICOM ảnh CT                 : {ct_df['ImageCount'].sum():>8,} ảnh")
print(f"• Dung lượng file chú thích XML (Toàn bộ)    : ~15 MB (~0.015 GB)")
print("="*75)
print(f"👉 KẾT LUẬN: Tập dữ liệu CT hoàn chỉnh nặng chính xác: {ct_only_gb:.2f} GB (Xấp xỉ ~120 GB).")
print("="*75)


In [ ]:
# Biểu đồ so sánh dung lượng
plt.figure(figsize=(9, 4))
categories = ["Toàn bộ LIDC-IDRI\n(Tất cả phương thức)", "Chỉ lấy CT Scans\n(1,018 Series)", "Dữ liệu không phải CT\n(DX, CR, SEG, SR)"]
sizes = [total_dataset_gb, ct_only_gb, non_ct_gb]
colors = ['#64748b', '#2563eb', '#94a3b8']

bars = plt.barh(categories, sizes, color=colors, height=0.5)
plt.xlabel("Dung lượng lưu trữ (Gigabytes - GB)")
plt.title("So sánh dung lượng lưu trữ: Toàn bộ Dataset vs. Chỉ lấy CT Scans", fontsize=12, fontweight="bold")
plt.xlim(0, 150)

for b in bars:
    plt.annotate(f'{b.get_width():.2f} GB',
                 (b.get_width() + 2, b.get_y() + b.get_height() / 2.),
                 va='center', fontweight='bold', fontsize=10)

plt.tight_layout()
plt.show()


---
## 7. Khảo sát phân bổ các đặc trưng của chuỗi ảnh CT (CT Feature Distribution)
Khảo sát phân bố của:
- **Số lượng lát cắt (ImageCount / Slices per scan)**: Ảnh hưởng trực tiếp đến bộ nhớ khi nạp vào kiến trúc **Inter-Slice Mamba**.
- **Dung lượng mỗi ca chụp (FileSize in MB)**: Ảnh hưởng đến tốc độ đọc đĩa (I/O throughput).
- **Mối tương quan giữa số lát cắt và dung lượng**.


In [ ]:
# 7. Thống kê mô tả chi tiết của CT Scans
img_stats = ct_df["ImageCount"].describe()
size_stats = ct_df["FileSize_MB"].describe()

stats_table = pd.DataFrame({
    "Chỉ số Thống kê": ["Số lượng mẫu (Count)", "Trung bình (Mean)", "Độ lệch chuẩn (Std)",
                        "Giá trị nhỏ nhất (Min)", "Phân vị 25% (Q1)", "Trung vị (Median / Q2)",
                        "Phân vị 75% (Q3)", "Giá trị lớn nhất (Max)"],
    "Số lát cắt (Slices)": [f"{img_stats['count']:.0f}", f"{img_stats['mean']:.1f}", f"{img_stats['std']:.1f}",
                            f"{img_stats['min']:.0f}", f"{img_stats['25%']:.0f}", f"{img_stats['50%']:.0f}",
                            f"{img_stats['75%']:.0f}", f"{img_stats['max']:.0f}"],
    "Dung lượng (MB)": [f"{size_stats['count']:.0f}", f"{size_stats['mean']:.1f} MB", f"{size_stats['std']:.1f} MB",
                        f"{size_stats['min']:.1f} MB", f"{size_stats['25%']:.1f} MB", f"{size_stats['50%']:.1f} MB",
                        f"{size_stats['75%']:.1f} MB", f"{size_stats['max']:.1f} MB"]
})

print(stats_table.to_string(index=False))


In [ ]:
# Biểu đồ phân bổ số lát cắt và dung lượng của 1,018 ca CT
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# 1. Histogram số lát cắt
sns.histplot(ct_df["ImageCount"], bins=40, kde=True, color="#2563eb", ax=axes[0, 0])
axes[0, 0].axvline(ct_df["ImageCount"].median(), color="red", linestyle="--", label=f"Median: {ct_df['ImageCount'].median():.0f} slices")
axes[0, 0].axvline(ct_df["ImageCount"].mean(), color="orange", linestyle=":", label=f"Mean: {ct_df['ImageCount'].mean():.1f} slices")
axes[0, 0].set_title("Phân bổ số lát cắt (ImageCount) mỗi ca CT", fontweight="bold")
axes[0, 0].set_xlabel("Số lượng lát cắt (Slices)")
axes[0, 0].set_ylabel("Số lượng ca CT")
axes[0, 0].legend()

# 2. Boxplot số lát cắt
sns.boxplot(x=ct_df["ImageCount"], color="#93c5fd", ax=axes[0, 1])
axes[0, 1].set_title("Biểu đồ hộp (Boxplot) phát hiện lát cắt ngoại lai (Outliers)", fontweight="bold")
axes[0, 1].set_xlabel("Số lượng lát cắt")

# 3. Histogram dung lượng MB
sns.histplot(ct_df["FileSize_MB"], bins=40, kde=True, color="#059669", ax=axes[1, 0])
axes[1, 0].axvline(ct_df["FileSize_MB"].median(), color="red", linestyle="--", label=f"Median: {ct_df['FileSize_MB'].median():.1f} MB")
axes[1, 0].set_title("Phân bổ dung lượng (MB) mỗi ca CT", fontweight="bold")
axes[1, 0].set_xlabel("Dung lượng (Megabytes)")
axes[1, 0].set_ylabel("Số lượng ca CT")
axes[1, 0].legend()

# 4. Tương quan giữa Số lát cắt và Dung lượng
sns.scatterplot(data=ct_df, x="ImageCount", y="FileSize_MB", hue="Manufacturer", alpha=0.7, ax=axes[1, 1], palette="tab10")
axes[1, 1].set_title("Tương quan tuyến tính giữa Số lát cắt & Dung lượng MB", fontweight="bold")
axes[1, 1].set_xlabel("Số lượng lát cắt (Slices)")
axes[1, 1].set_ylabel("Dung lượng (MB)")

plt.tight_layout()
plt.show()


---
## 8. Khảo sát đa dạng thiết bị chụp (Scanner Diversity & Domain Bias)
Trong Medical AI, **độ sai lệch thiết bị (Scanner Bias)** là thách thức lớn nhất đối với khả năng khái quát hóa (Generalization) của mô hình học sâu.
Khảo sát các hãng sản xuất thiết bị CT và các dòng máy phổ biến nhất có trong LIDC-IDRI.


In [ ]:
# 8. Phân bổ Hãng sản xuất (Manufacturer) & Dòng máy (Model)
mfg_counts = ct_df["Manufacturer"].value_counts()
print("Phân bổ Hãng sản xuất máy chụp CT:")
for mfg, count in mfg_counts.items():
    print(f"  • {mfg:<22s}: {count:>4} ca CT ({count/len(ct_df)*100:>5.1f}%)")

top_models = ct_df["ManufacturerModelName"].value_counts().head(10)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Biểu đồ hãng máy
mfg_counts.plot(kind="bar", color=['#1e40af', '#0284c7', '#0d9488', '#ea580c'], ax=axes[0], rot=15)
axes[0].set_title("Phân bổ Hãng sản xuất máy chụp CT (GE áp đảo ~66%)", fontsize=11, fontweight="bold")
axes[0].set_ylabel("Số lượng ca CT")
for p in axes[0].patches:
    axes[0].annotate(f'{p.get_height()}', (p.get_x() + p.get_width() / 2., p.get_height()),
                    ha='center', va='bottom', fontsize=9)

# Biểu đồ Top 10 Model máy
top_models.sort_values().plot(kind="barh", color="#6366f1", ax=axes[1])
axes[1].set_title("Top 10 Dòng máy chụp CT phổ biến nhất trong LIDC-IDRI", fontsize=11, fontweight="bold")
axes[1].set_xlabel("Số lượng ca CT")

plt.tight_layout()
plt.show()


---
## 9. Chiến lược tải dữ liệu & Dự toán thời gian băng thông
Thay vì tải một lần toàn bộ 120 GB (dễ bị đứt kết nối mạng hoặc đầy bộ nhớ Google Colab), chúng ta chia thành **4 giai đoạn nghiên cứu**.


In [ ]:
# 9. Bảng chiến lược tải và dự toán thời gian
strategies = [
    {"Giai đoạn": "GĐ 1: Smoke Test / Pipeline Check", "Số ca CT": 10, "Dung lượng (GB)": 1.20, "Mục đích": "Kiểm thử I/O pipeline, tiền xử lý, trích xuất mask"},
    {"Giai đoạn": "GĐ 2: Baseline MambaVision (Proof of Concept)", "Số ca CT": 100, "Dung lượng (GB)": 11.96, "Mục đích": "Huấn luyện thử nghiệm MambaVision-T, tinh chỉnh hyperparams"},
    {"Giai đoạn": "GĐ 3: Nghiên cứu Ablation Study (3D-Aware)", "Số ca CT": 500, "Dung lượng (GB)": 59.80, "Mục đích": "So sánh 2.5D vs 3D-aware vs Inter-Slice Mamba"},
    {"Giai đoạn": "GĐ 4: Full-scale Dataset", "Số ca CT": 1018, "Dung lượng (GB)": 119.59, "Mục đích": "Huấn luyện toàn diện và benchmark kết quả báo cáo niên luận"}
]

df_strat = pd.DataFrame(strategies)

# Dự toán thời gian tải theo tốc độ mạng
def est_time(gb, mbps):
    mb = gb * 1024
    speed_mb_s = mbps / 8.0
    seconds = mb / speed_mb_s
    if seconds < 60:
        return f"{seconds:.0f} giây"
    elif seconds < 3600:
        return f"{seconds/60:.1f} phút"
    else:
        return f"{seconds/3600:.1f} giờ"

df_strat["Thời gian (50 Mbps)"] = df_strat["Dung lượng (GB)"].apply(lambda x: est_time(x, 50))
df_strat["Thời gian (100 Mbps)"] = df_strat["Dung lượng (GB)"].apply(lambda x: est_time(x, 100))
df_strat["Thời gian (Google Colab ~500 Mbps)"] = df_strat["Dung lượng (GB)"].apply(lambda x: est_time(x, 500))

print("="*110)
print(f"{'CHIẾN LƯỢC TẢI DỮ LIỆU TỪNG BƯỚC & DỰ TOÁN THỜI GIAN':^110}")
print("="*110)
print(df_strat.to_string(index=False))
print("="*110)


---
## 10. Tổng kết các con số then chốt & Khuyến nghị cho Đề tài Niên luận

### 📌 1. Kết luận các câu hỏi cốt lõi:
* **Số lượng bệnh nhân có CT**: **1,010 / 1,010 bệnh nhân (100%)** đều có ít nhất 1 ca chụp CT. Tổng cộng có **1,018 series CT** (8 bệnh nhân chụp 2 lần).
* **Dung lượng CT thuần túy**: **119.59 GB** (Xấp xỉ **120 GB**) gồm **243,958 lát cắt DICOM**.
* **Dung lượng tiết kiệm được**: Lọc bỏ các ảnh X-quang (DX, CR) và hàng nghìn file SEG/SR giúp tiết kiệm được **~7.76 GB** và loại bỏ hơn **14,000 HTTP requests** không cần thiết.
* **Đặc trưng thể tích**: Trung bình mỗi ca CT có **239.6 lát cắt** (Median: 203.5 lát, Min: 65, Max: 764 lát).

---

### 🚀 2. Khuyến nghị thiết kế mô hình MambaVision & Inter-Slice Mamba:
1. **Quản lý biến thiên số lát cắt (65 đến 764 lát)**:
   - Vì số lát cắt dao động lớn, **không nên nạp toàn bộ 764 lát vào GPU cùng một lúc**.
   - Nên áp dụng cơ chế **Sliding Window 2.5D** (ví dụ: cửa sổ $k = 5$ hoặc $k = 9$ lát lân cận) đưa qua MambaVision Backbone để trích xuất spatial features.
   - Sau đó dùng **Inter-Slice Mamba (Selective State Space Model)** truyền trạng thái ẩn (hidden states) xuôi/ngược theo trục $Z$ từ lát cắt đầu đến lát cắt cuối.
2. **Xử lý Scanner Bias**:
   - Máy chụp của **GE chiếm 65.8%**, **Siemens chiếm 20.1%**, **Philips 7.3%**, **Toshiba 6.8%**.
   - Khi chia tập Train/Validation/Test, cần **Stratified Split theo Manufacturer** để đảm bảo mô hình không bị học vẹt theo đặc tính của riêng máy GE.
